# Image preprocessing analysis

## 1. Notebook setup

### 1.1. Imports

In [1]:
import os

import pydiacom
import numpy as np
import pandas as pd

import configuration as config

ModuleNotFoundError: No module named 'pydiacom'

### 1.2. Configuration

## 2. Load series metadata

In [ ]:
series_df = pd.read_csv(config.TRAIN_SERIES_CSV)

## 2. Reference series selection

### 2.1. Selection function

In [ ]:
def find_best_atlas_candidate(dataset_root_dir, target_plane="SAGITTAL", target_sensitivity="FLUID_SENSITIVE"):
    """
    Scans the dataset to find the most 'median' patient sequence to act as an atlas.
    It looks for standard resolution, slice count, and voxel spacing.
    """

    candidates = []

    # Iterate through your patient directories
    for root, dirs, files in os.walk(dataset_root_dir):
    
        # Filter for folders that contain DICOM slices
        dicom_files = [f for f in files if f.lower().endswith('.dcm') or f.isdigit()]

        if not dicom_files:
            continue
            
        try:
            # Read metadata from the first slice in the directory
            sample_slice = pydicom.dcmread(os.path.join(root, dicom_files[0]))
            
            # extract features (Map these to your specific DICOM tags)
            # e.g., ImageType, SeriesDescription, or custom attributes
            plane = sample_slice.get("SeriesDescription", "").upper() 
            num_slices = len(dicom_files)
            rows = sample_slice.Rows          # Matrix Height
            cols = sample_slice.Columns       # Matrix Width
            pixel_spacing = sample_slice.PixelSpacing # Voxel resolution [dx, dy]
            
            # Check if this folder matches the plane we are looking for
            if target_plane in plane:
                candidates.append({
                    "dir_path": root,
                    "num_slices": num_slices,
                    "matrix_shape": (rows, cols),
                    "pixel_spacing": (float(pixel_spacing[0]), float(pixel_spacing[1]))
                })
        except Exception:
            continue

    if not candidates:
        raise ValueError(f"No series found matching {target_plane}")

    # --- SELECT THE MEDIAN EXTRACTION PATH ---
    # We want a scan that is right in the middle of our dataset distribution
    slice_counts = [c["num_slices"] for c in candidates]
    median_slices = np.median(slice_counts)
    
    # Sort candidates by how close they are to the median slice count
    # and standard matrix sizes (e.g., 512x512)
    candidates.sort(key=lambda x: abs(x["num_slices"] - median_slices))
    
    best_candidate = candidates[0]
    
    print(f"--- Found Best Atlas for {target_plane} ---")
    print(f"Directory: {best_candidate['dir_path']}")
    print(f"Slices: {best_candidate['num_slices']} (Dataset Median: {median_slices})")
    print(f"Resolution: {best_candidate['matrix_shape']}")
    
    return best_candidate["dir_path"]